In [1]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque

### Neural Network

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DQN(nn.Module):

    def __init__(self, state_dim, action_dim):
        super().__init__()

        self.fc1 = nn.Linear(state_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.out = nn.Linear(128, action_dim)

    def forward(self, x):

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        return self.out(x)

### Replay buffer

In [5]:
import random
from collections import deque
import numpy as np


class ReplayBuffer:

    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):

        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):

        batch = random.sample(self.buffer, batch_size)

        states, actions, rewards, next_states, dones = zip(*batch)

        return (
            np.array(states),
            np.array(actions),
            np.array(rewards),
            np.array(next_states),
            np.array(dones),
        )

    def __len__(self):
        return len(self.buffer)

### Training script

In [9]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np


env = gym.make("CartPole-v1")

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n


# hyperparameters
gamma = 0.99
lr = 1e-3
batch_size = 64
episodes = 500
target_update = 10

epsilon = 1.0
epsilon_min = 0.01
epsilon_decay = 0.995


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


online_net = DQN(state_dim, action_dim).to(device)
target_net = DQN(state_dim, action_dim).to(device)

target_net.load_state_dict(online_net.state_dict())
target_net.eval()


optimizer = optim.Adam(online_net.parameters(), lr=lr)
loss_fn = nn.MSELoss()

buffer = ReplayBuffer()


def select_action(state):

    global epsilon

    if np.random.rand() < epsilon:
        return env.action_space.sample()

    state = torch.FloatTensor(state).unsqueeze(0).to(device)

    with torch.no_grad():
        q_values = online_net(state)

    return torch.argmax(q_values).item()


def train_step():

    if len(buffer) < batch_size:
        return

    states, actions, rewards, next_states, dones = buffer.sample(batch_size)

    states = torch.FloatTensor(states).to(device)
    actions = torch.LongTensor(actions).to(device)
    rewards = torch.FloatTensor(rewards).to(device)
    next_states = torch.FloatTensor(next_states).to(device)
    dones = torch.FloatTensor(dones).to(device)


    q_values = online_net(states)
    next_q_values = target_net(next_states)


    q_value = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)

    next_q_value = torch.max(next_q_values, 1)[0]

    expected_q = rewards + gamma * next_q_value * (1 - dones)


    loss = loss_fn(q_value, expected_q.detach())


    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


for episode in range(episodes):

    state, _ = env.reset()

    total_reward = 0
    done = False


    while not done:

        action = select_action(state)

        next_state, reward, terminated, truncated, _ = env.step(action)

        done = terminated or truncated

        buffer.push(state, action, reward, next_state, done)

        state = next_state
        total_reward += reward

        train_step()


    if epsilon > epsilon_min:
        epsilon *= epsilon_decay


    if episode % target_update == 0:
        target_net.load_state_dict(online_net.state_dict())


    print(f"Episode {episode} Reward {total_reward} Epsilon {epsilon:.3f}")


torch.save(online_net.state_dict(), "dqn_cartpole.pth")

print("Model saved!")

env.close()

Episode 0 Reward 16.0 Epsilon 0.995
Episode 1 Reward 20.0 Epsilon 0.990
Episode 2 Reward 36.0 Epsilon 0.985
Episode 3 Reward 16.0 Epsilon 0.980
Episode 4 Reward 15.0 Epsilon 0.975
Episode 5 Reward 15.0 Epsilon 0.970
Episode 6 Reward 24.0 Epsilon 0.966
Episode 7 Reward 24.0 Epsilon 0.961
Episode 8 Reward 17.0 Epsilon 0.956
Episode 9 Reward 32.0 Epsilon 0.951
Episode 10 Reward 13.0 Epsilon 0.946
Episode 11 Reward 11.0 Epsilon 0.942
Episode 12 Reward 20.0 Epsilon 0.937
Episode 13 Reward 11.0 Epsilon 0.932
Episode 14 Reward 12.0 Epsilon 0.928
Episode 15 Reward 36.0 Epsilon 0.923
Episode 16 Reward 17.0 Epsilon 0.918
Episode 17 Reward 29.0 Epsilon 0.914
Episode 18 Reward 18.0 Epsilon 0.909
Episode 19 Reward 15.0 Epsilon 0.905
Episode 20 Reward 18.0 Epsilon 0.900
Episode 21 Reward 17.0 Epsilon 0.896
Episode 22 Reward 13.0 Epsilon 0.891
Episode 23 Reward 31.0 Epsilon 0.887
Episode 24 Reward 15.0 Epsilon 0.882
Episode 25 Reward 39.0 Epsilon 0.878
Episode 26 Reward 14.0 Epsilon 0.873
Episode 27 

### Evaluation

In [12]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import torch


# create environment
env = gym.make("CartPole-v1", render_mode="rgb_array")

# record video
env = RecordVideo(
    env,
    video_folder="videos",
    episode_trigger=lambda episode_id: True
)


state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n


# load trained model
model = DQN(state_dim, action_dim)
model.load_state_dict(torch.load("dqn_cartpole.pth"))
model.eval()


state, _ = env.reset()

done = False
total_reward = 0


while not done:

    state_tensor = torch.FloatTensor(state).unsqueeze(0)

    with torch.no_grad():
        q_values = model(state_tensor)

    action = torch.argmax(q_values).item()

    state, reward, terminated, truncated, _ = env.step(action)

    done = terminated or truncated

    total_reward += reward


env.close()

print("Episode reward:", total_reward)
print("Video saved in ./videos/")

/home/sachchida/anaconda3/envs/rl/lib/python3.11/site-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /home/sachchida/gitrepo/reinforcementLearning/ipynb/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
/home/sachchida/anaconda3/envs/rl/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Episode reward: 500.0
Video saved in ./videos/
